# Calibrated Bin50 B-lineage RNA coverage

This report documents the approved primary definition: patient/UMI-calibrated q90 extended-panel score, at least two informative extended genes detected in the center-excluded 50 µm neighborhood, and at least five connected positive Bin50. Components of at least three bins are retained as a sensitivity result.

The endpoint is **B-lineage RNA coverage**, not B-cell identity, a B-cell niche, an aggregate, or a TLS call. Disease labels were not used to construct or calibrate the mask.

## Reproducible render command

The exact command is stored in `scripts/1054_render_b_lineage_coverage_report.sh`. It generates a `YYMMDDHHMMSS` timestamp, sets the two input directories as environment variables, executes this notebook with nbconvert, and writes the HTML to `results/` and the render log to `logs/`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys

import matplotlib
import pandas as pd
from IPython.display import Image, Markdown, display

project_root = Path(os.environ['B_COVERAGE_PROJECT_ROOT']).resolve()
result_dir = Path(os.environ['B_COVERAGE_RESULT_DIR']).resolve()
rule_audit_dir = Path(os.environ['B_COVERAGE_RULE_AUDIT_DIR']).resolve()
assert result_dir.exists(), result_dir
assert rule_audit_dir.exists(), rule_audit_dir

chip_summary = pd.read_csv(result_dir / '04_PRIMARY_MASK_CHIP_SUMMARY.tsv', sep='\t')
spatial = pd.read_csv(result_dir / '02_SPATIAL_RULE_COMPARISON.tsv', sep='\t')
patient = pd.read_csv(result_dir / '03_PATIENT_COVERAGE_SUMMARY.tsv', sep='\t')
audit = pd.read_csv(result_dir / 'qc/05_FINAL_MASK_AUDIT.tsv', sep='\t')
genes = pd.read_csv(rule_audit_dir / '04_GENE_INFORMATIVENESS_AUDIT.tsv', sep='\t')
session = json.loads((result_dir / 'SESSION_INFO.json').read_text(encoding='utf-8'))
assert audit['passed'].all()
display(Markdown(f'**Final result directory:** `{result_dir.relative_to(project_root)}`'))

## Frozen definitions and audit

In [ ]:
display(pd.DataFrame({'item': session.keys(), 'value': [str(value) for value in session.values()]}))
display(audit)

## Informative extended genes

In [ ]:
extended = genes[genes['panel'].eq('B__extended_only')].copy()
informative = extended[extended['informative_for_bin_rule']].sort_values('median_matched_score_delta', ascending=False)
display(Markdown(f'**Informative extended genes retained:** {len(informative)}'))
display(informative[['gene', 'evaluable_patient_n', 'positive_score_delta_patient_n', 'positive_chip_n', 'median_matched_score_delta', 'median_matched_detection_delta']])

## Primary q90, component ≥5 result

In [ ]:
display(chip_summary)
totals = chip_summary.select_dtypes('number').sum()
summary_lines = [
    f"Candidate Bin50: {int(totals['candidate_bin_n']):,}",
    f"q90 bin-supported before connectivity: {int(totals['calibrated_bin_support_n']):,}",
    f"Primary region-supported (component >=5): {int(totals['primary_region_support_n']):,}",
    f"Extended gain beyond the core-only comparator: {int(totals['extended_gain_n']):,}",
    f"Primary bins with core corroboration: {int(totals['core_corroborated_n']):,}",
]
display(Markdown('\n'.join(f'- {line}' for line in summary_lines)))

## Component ≥3 sensitivity and q95 sensitivity

In [ ]:
comparison = (spatial.groupby('background_quantile', as_index=False)[['bin_n', 'called_n', 'region_supported_ge3_n', 'region_supported_ge5_n']].sum())
display(comparison)
q90 = comparison[comparison['background_quantile'].eq(0.90)].iloc[0]
display(Markdown(f"At q90, raising the connectivity criterion from ≥3 to ≥5 changes the retained mask from **{int(q90['region_supported_ge3_n']):,}** to **{int(q90['region_supported_ge5_n']):,}** Bin50."))

## Patient-level coverage

In [ ]:
patient_q90 = patient[patient['background_quantile'].eq(0.90)].sort_values(['chip', 'patient'])
display(patient_q90[['chip', 'patient', 'candidate_bin_n', 'called_fraction', 'region_supported_ge3_fraction', 'region_supported_ge5_fraction']])

## Spatial review

In [ ]:
for chip in ['J2', 'L3', 'Y40102K8']:
    image_path = result_dir / 'figures' / f'02_{chip}_Q90_Q95_SPATIAL_OVERVIEW_EN.png'
    display(Markdown(f'### {chip}'))
    display(Image(filename=str(image_path), width=1000))

## Interpretation boundary

The extended panel is used to increase detectable B-lineage RNA coverage in low-UMI spatial data. The UMI-matched q90 requirement limits calls driven by depth alone, the multi-gene requirement limits single broad-gene calls, and the five-bin connectivity criterion defines a spatial region rather than an isolated signal. Core genes are retained as corroboration and do not veto extended support. The result must not be interpreted as a pure B-cell label, cell count, aggregate, niche, or TLS annotation.

## Session information

In [ ]:
session_info = {
    'python': sys.version.replace('\n', ' '),
    'platform': platform.platform(),
    'pandas': pd.__version__,
    'matplotlib': matplotlib.__version__,
}
display(pd.DataFrame({'component': session_info.keys(), 'version': session_info.values()}))